# Week 5: Uplift Modeling with R-Learner (Double Machine Learning)
Trong notebook này, chúng ta sẽ xây dựng mô hình R-Learner để giải quyết bài toán Causal Inference (dự đoán CATE). 
Khác với T-Learner hay S-Learner, R-Learner tập trung vào việc "residualize" (khử nhiễu nền) Outcome và Treatment trước khi học tín hiệu nhân quả.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 1. Tải dữ liệu
data_path = '../../data/processed/segmented_simulation_data.csv'
df = pd.read_csv(data_path)
print(f"Tổng số User: {len(df):,}")

# Các đặc trưng (Features)
features = ['age', 'is_urban', 'preferred_hour', 'is_rush_hour', 'is_airport_trip', 'is_rain_rider', 'is_weekend_rider', 'is_credit_card', 'passenger_count', 'monthly_rides_history', 'recency_days']

X = df[features]
y = df['Y_rand']
T = df['treatment_rand']

# Chia tập Train/Validation/Test (60/20/20)
X_tv, X_test, y_tv, y_test, T_tv, T_test = train_test_split(X, y, T, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val, T_train, T_val = train_test_split(X_tv, y_tv, T_tv, test_size=0.25, random_state=42)

df_test = df.loc[X_test.index].copy()
df_test['avg_fare'] = df_test['avg_fare_per_trip']
print(f"Test set size: {len(df_test):,}")

Tổng số User: 20,000
Test set size: 4,000


## Bước 1: Huấn luyện mô hình cơ sở (Base Outcome) và Khử nhiễu
Dự đoán số chuyến đi tự nhiên mà không phụ thuộc vào Voucher.

In [2]:
params_y = dict(random_state=42, n_estimators=100, learning_rate=0.05, max_depth=4)
m_y = xgb.XGBRegressor(**params_y)
m_y.fit(X_train, y_train)

# Tính Residuals
p = 0.5  # Do RCT 50/50
Y_tilde = y_train - m_y.predict(X_train)
T_tilde = T_train - p

## Bước 2: Học tín hiệu nhân quả (CATE) trên Residuals
Trọng số học được tối ưu hóa dựa trên phương sai của Treatment.

In [3]:
uplift_target = Y_tilde / T_tilde
weights = T_tilde ** 2

params_cate = dict(random_state=42, n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=10)
r_model = xgb.XGBRegressor(**params_cate)
r_model.fit(X_train, uplift_target, sample_weight=weights)

# Dự đoán CATE trên Test set
cate_pred = r_model.predict(X_test)
df_test['cate_pred'] = cate_pred
df_test['pred_rides_treated'] = m_y.predict(X_test) + 0.5 * cate_pred

print(f"Mean predicted CATE: {cate_pred.mean():.4f}")

Mean predicted CATE: 0.9551


## Bước 4: Đánh giá bằng Qini Curve & AUUC
Đường cong Qini đo lường khả năng Xếp hạng (Ranking) khách hàng của R-Learner. Đường cong càng vồng lên cao so với đường Random (phát ngẫu nhiên), mô hình càng có khả năng nhắm mục tiêu chính xác những người nhạy cảm với khuyến mãi.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
df_res = pd.DataFrame({
    'y_true': df_test['Y_rand'],
    'T': df_test['treatment_rand'],
    'CATE_pred': df_test['cate_pred']
})

def plot_qini_curve(df_res, title='Qini Curve (R-Learner Champion)'):
    df_sorted = df_res.sort_values(by='CATE_pred', ascending=False).reset_index(drop=True)
    df_sorted['T_count'] = df_sorted['T'].cumsum()
    df_sorted['C_count'] = (1 - df_sorted['T']).cumsum()
    df_sorted['Y_T_sum'] = (df_sorted['y_true'] * df_sorted['T']).cumsum()
    df_sorted['Y_C_sum'] = (df_sorted['y_true'] * (1 - df_sorted['T'])).cumsum()
    df_sorted['C_count'] = df_sorted['C_count'].replace(0, 1)
    df_sorted['Qini'] = df_sorted['Y_T_sum'] - df_sorted['Y_C_sum'] * (df_sorted['T_count'] / df_sorted['C_count'])
    total_uplift = df_sorted['Qini'].iloc[-1]
    df_sorted['Random_Qini'] = np.linspace(0, total_uplift, len(df_sorted))
    auuc_model = np.trapz(df_sorted['Qini'], dx=1)
    auuc_random = np.trapz(df_sorted['Random_Qini'], dx=1)
    qini_coef = (auuc_model - auuc_random) / auuc_random if auuc_random != 0 else 0
    plt.figure(figsize=(10, 6))
    plt.plot(df_sorted.index / len(df_sorted) * 100, df_sorted['Qini'], label=f'R-Learner (Qini Coef: {qini_coef:.3f})', color='#00E5FF', linewidth=3)
    plt.plot(df_sorted.index / len(df_sorted) * 100, df_sorted['Random_Qini'], label=f'Random (Qini Coef: 0.000)', color='gray', linestyle='--')
    plt.title(title, fontsize=16, fontweight='bold')
    plt.xlabel('% Targeted Population', fontsize=12)
    plt.ylabel('Cumulative Incremental Rides', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(alpha=0.3)
    plt.show()

plot_qini_curve(df_res)


## Bước 5: Đánh giá Kinh tế & Policy Simulator
Chuyển đổi CATE (số chuyến đi tăng thêm) thành Giá trị Kinh tế (Expected Incremental Profit).
Công thức: $EV = CATE \times Margin - Predicted\_Rides \times Voucher\_Cost$

In [4]:
VOUCHER_RATE = 0.15
MARGIN_RATE = 0.70
CAMPAIGN_BUDGET = 50000

df_test['voucher_cost'] = df_test['avg_fare'] * VOUCHER_RATE
df_test['margin_per_ride'] = df_test['avg_fare'] * MARGIN_RATE
df_test['expected_value'] = (df_test['cate_pred'] * df_test['margin_per_ride']) - (df_test['pred_rides_treated'] * df_test['voucher_cost'])

if 'cate_true' in df_test.columns:
    df_test['oracle_ev'] = (df_test['cate_true'] * df_test['margin_per_ride']) - (df_test['pred_rides_treated'] * df_test['voucher_cost'])

def evaluate_policy(mask, name):
    targeted = df_test[mask]
    n_t = mask.sum()
    if n_t == 0: return {"Policy": name, "Users": 0, "Predicted Profit": 0, "Oracle Profit": 0}
    
    pred_ev = targeted['expected_value'].sum()
    gt_ev = targeted['oracle_ev'].sum() if 'oracle_ev' in df_test.columns else pred_ev
    return {"Policy": name, "Users": int(n_t), "Predicted Profit": round(pred_ev, 0), "Oracle Profit": round(gt_ev, 0)}

policies = []
# 1. Mass Voucher
policies.append(evaluate_policy(df_test['cate_pred'].notnull(), "1. Mass Voucher"))
# 2. Segment (Suburban)
policies.append(evaluate_policy(df_test['persona'].str.contains('Suburban', na=False), "2. Segment Targeting (Suburban)"))
# 3. Profit Targeting (EV > 0)
policies.append(evaluate_policy(df_test['expected_value'] > 0, "3. AI Profit Targeting"))
# 4. Oracle
if 'cate_true' in df_test.columns:
    policies.append(evaluate_policy(df_test['oracle_ev'] > 0, "4. Oracle Targeting"))

pd.DataFrame(policies)

,Policy,Users,Predicted Profit,Oracle Profit
0,1. Mass Voucher,4000,-65358.0,-76313.0
1,2. Segment Targeting (Suburban),1660,-4286.0,-996.0
2,3. AI Profit Targeting,888,7939.0,7060.0
3,4. Oracle Targeting,1002,4971.0,10818.0
